# CDR-MLC — paper pipeline on regenerated data

پیاده‌سازی از ابتدا بر اساس بخش‌های ۴٫۳ و ۴٫۴ و الگوریتم‌های ۱ و ۲ فایل `CDR-MLC(2).pdf`.

**Paper requirements:** 3 timing features → 5 trailing-window statistics each → MiniBatchKMeans(k=3) → 3 multiclass Random Forest experts. Experts exclude the 3 raw timing features and all 15 routing statistics. RF experts: 20 trees (§5.1); reference RF: 100 trees. Scenarios 1–3 follow Table 4.

**Explicit assumptions / limitations:** paper does not enumerate all 35 input fields or specify window length, ddof, warm-up, scaling, MBK batch size or full RF hyperparameters. Here window=3 follows the established project setting; ddof=0; complete trailing windows only; StandardScaler for routing; explicit MBK settings below. Available numeric fields plus Flgs/State/TcpOpt are used; dimensions are reported, never padded to 35. Empty/constant/identical fields and suspicious IdleTime are excluded using training-only rules. This reproduces the architecture with documented adaptations, not the original dataset or published scores.

Sequence boundaries are source capture files, reflecting independently collected runs. In deployment a sequence ID must be externally available (sensor/host/run), never inferred from the unknown true application label. One source file may contain multiple sessions; Argus Start/Status rows remain records, not independent connections. A shared sliding window across arbitrary simultaneous classes would be a different experiment.

No test labels, test congestion levels, or test-fitted scaling participate in routing or prediction. Clusters are unlabeled regimes, not automatically Low/Medium/High, particularly when training on just Low.

Paper PDF SHA256: `c449099f7e32dea62e6059f4104cd564c4c299482397ae618e5d6b021194d461`.


In [ ]:
from pathlib import Path
from collections import deque
import hashlib, json, platform, re, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from sklearn.cluster import MiniBatchKMeans
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
    classification_report, confusion_matrix, ConfusionMatrixDisplay, silhouette_score)
from IPython.display import display

SEED=42
WINDOW=3
TIMING=['TcpRtt','SynAck','AckDat']
STATS=['mean','max','median','min','std']
CLASSES=['HTTP','SFTP','SMTP','SSH','Video']
MBK_CONFIG=dict(n_clusters=3,batch_size=1024,n_init=10,max_iter=100,random_state=SEED)
RF_CONFIG=dict(n_estimators=20,criterion='gini',max_depth=None,min_samples_leaf=1,
               max_features='sqrt',bootstrap=True,class_weight=None,n_jobs=-1,random_state=SEED)
RUN_REFERENCE_RF=True
DATA_DIR_OVERRIDE=None
bases=[Path.cwd(),*Path.cwd().parents]
candidates=[b/r for b in bases for r in ['CDR_MLC/DATASETS/CDR-MLC/New_Version','DATASETS/CDR-MLC/New_Version']]
DATA_DIR=Path(DATA_DIR_OVERRIDE) if DATA_DIR_OVERRIDE else next((p for p in candidates if p.is_dir()),None)
if DATA_DIR is None: raise FileNotFoundError('Set DATA_DIR_OVERRIDE')
OUT=DATA_DIR.parents[2]/'outputs'/'paper_implementation'
OUT.mkdir(parents=True,exist_ok=True)
print('Data:',DATA_DIR,'\nOutput:',OUT)
print('Versions:',platform.python_version(),pd.__version__,np.__version__,sklearn.__version__)


## 1. Validate provenance, endpoints and fields
Labels come from the capture manifest (filename); original files are never modified. Non-service records are excluded with a report. SSH and SFTP share endpoints and port: their labels rely on known collection runs, not port inference.
Reverse-oriented records trigger an explicit error instead of silently mixing source/destination features. Timestamps are used for order only. Zeros are preserved; negative/missing timing records cannot provide routing windows and are reported.


In [ ]:
SERVICES={'HTTP':('192.168.2.122',8080),'SFTP':('192.168.2.120',22),
          'SMTP':('192.168.2.120',8025),'SSH':('192.168.2.120',22),'Video':('192.168.2.121',5000)}
CLIENT='192.168.1.111'
frames=[];audit=[]
for p in sorted(DATA_DIR.glob('*.flow')):
    m=re.fullmatch(r'(HTTP|SFTP|SMTP|SSH|Video)_(Low|Medium|High)',p.stem)
    if not m: raise ValueError(f'Unexpected capture: {p.name}')
    label,level=m.groups(); server,port=SERVICES[label]
    d=pd.read_csv(p,low_memory=False,on_bad_lines='error')
    d.columns=d.columns.str.strip()
    required=['StartTime','SrcAddr','DstAddr','Proto','Sport','Dport',*TIMING]
    if set(required)-set(d):raise ValueError(f'Missing columns in {p.name}')
    for c in d.select_dtypes('object'):d[c]=d[c].str.strip().replace('',np.nan)
    d['source_row']=np.arange(len(d))+2
    tcp=d.Proto.eq('tcp')
    forward=tcp & d.SrcAddr.eq(CLIENT) & d.DstAddr.eq(server) & pd.to_numeric(d.Dport,errors='coerce').eq(port)
    reverse=tcp & d.SrcAddr.eq(server) & d.DstAddr.eq(CLIENT) & pd.to_numeric(d.Sport,errors='coerce').eq(port)
    if reverse.any():raise ValueError(f'{p.name}: reverse-oriented records require canonicalization before analysis')
    audit.append(dict(file=p.name,raw_records=len(d),retained=int(forward.sum()),excluded=int((~forward).sum()),
                      sha256=hashlib.sha256(p.read_bytes()).hexdigest()))
    d=d.loc[forward].copy()
    if d.empty:raise ValueError(f'No service records in {p.name}')
    d['timestamp']=pd.to_datetime(d.StartTime,format='%Y/%m/%d %H:%M:%S.%f',errors='raise')
    for c in TIMING:d[c]=pd.to_numeric(d[c],errors='coerce').replace([np.inf,-np.inf],np.nan)
    d['traffic_label']=label;d['congestion_level']=level;d['sequence_id']=p.stem;d['source_file']=p.name
    frames.append(d.sort_values(['timestamp','source_row'],kind='stable'))
data=pd.concat(frames,ignore_index=True)
assert set(zip(data.traffic_label,data.congestion_level))=={(c,l) for c in CLASSES for l in ['Low','Medium','High']}
aud=pd.DataFrame(audit);display(aud);aud.to_csv(OUT/'input_audit.csv',index=False)
display(data.groupby(['traffic_label','congestion_level']).size().unstack().reindex(columns=['Low','Medium','High']))


## 2. Causal trailing statistics — no cross-run windows
A row's window contains itself and its previous two records within the same capture. Invalid timing resets the window; no interpolation or future records are used. Warm-up drops are reported. The streaming and batch transformations below are checked for equality.


In [ ]:
TREND_COLUMNS=[f'{f}_{s}' for f in TIMING for s in STATS]
def trend_windows(frame):
    chunks=[];report=[]
    for sequence,g in frame.groupby('sequence_id',sort=False):
        g=g.sort_values(['timestamp','source_row'],kind='stable').copy()
        valid=np.isfinite(g[TIMING]).all(axis=1)&g[TIMING].ge(0).all(axis=1)
        segment=(~valid).cumsum()
        pieces=[]
        for _,part in g.loc[valid].groupby(segment[valid],sort=False):
            z=part.copy()
            for f in TIMING:
                roll=part[f].rolling(WINDOW,min_periods=WINDOW)
                for stat in STATS:z[f'{f}_{stat}']=roll.std(ddof=0) if stat=='std' else getattr(roll,stat)()
            pieces.append(z.dropna(subset=TREND_COLUMNS))
        kept=pd.concat(pieces) if pieces else g.iloc[:0].assign(**{c:np.nan for c in TREND_COLUMNS})
        report.append(dict(sequence_id=sequence,input_records=len(g),invalid_timing=int((~valid).sum()),
                           warmup=int(valid.sum()-len(kept)),usable=len(kept)))
        chunks.append(kept)
    return pd.concat(chunks,ignore_index=True),pd.DataFrame(report)

class StreamTrend:
    def __init__(self):self.buffers={}
    def update(self,sequence_id,values):
        a=np.asarray(values,dtype=float)
        q=self.buffers.setdefault(sequence_id,deque(maxlen=WINDOW))
        if not np.isfinite(a).all() or (a<0).any():q.clear();return None
        q.append(a)
        if len(q)<WINDOW:return None
        x=np.asarray(q)
        return np.array([v for j in range(3) for v in (x[:,j].mean(),x[:,j].max(),np.median(x[:,j]),x[:,j].min(),x[:,j].std(ddof=0))])

# Verify ordering, warm-up and the exact vector used by inference.
probe=data.groupby('sequence_id',sort=False).head(20).copy()
batch,_=trend_windows(probe)
s=StreamTrend(); stream=[]
for _,r in probe.iterrows():
    v=s.update(r.sequence_id,r[TIMING].to_numpy(float))
    if v is not None:stream.append(v)
assert np.allclose(np.asarray(stream),batch[TREND_COLUMNS].to_numpy(),atol=1e-10)
print('Streaming/batch window equivalence passed.')


## 3. Fit on the training domain only
Choose available features using training data only. Keep numeric behavioral fields, not identifiers/labels/time/TTL/export cause. Encode Flgs, State, TcpOpt with train-fitted one-hot encoding (unknown test categories ignored). Record every excluded field and reason.

The PDF's exact 35-column schema is not provided. The resulting dimensions therefore depend on the current exports. No features are selected from test performance. IdleTime is excluded because the audit found a per-file constant around 1.79e9, and the project explicitly excludes it.


In [ ]:
META={'StartTime','SrcAddr','DstAddr','Proto','Sport','Dport','Label','Cause','Dir','sTtl','dTtl',
      'traffic_label','congestion_level','sequence_id','source_file','source_row','timestamp'}
CATEGORICAL=['Flgs','State','TcpOpt']
def select_columns(train):
    nums=[];cats=[];reasons=[]
    for c in train.columns:
        reason=None
        if c in META or c in TREND_COLUMNS:reason='metadata_or_routing_statistics'
        elif re.search(r'\.\d+$',c):reason='duplicate_header'
        elif c=='IdleTime':reason='unresolved_semantics'
        elif c in CATEGORICAL:
            if train[c].nunique(dropna=True)>1:cats.append(c)
            else:reason='empty_or_constant'
        else:
            n=pd.to_numeric(train[c],errors='coerce').replace([np.inf,-np.inf],np.nan)
            if not n.notna().any():reason='empty_or_non_numeric'
            elif n.nunique()<2:reason='constant'
            elif any(n.equals(pd.to_numeric(train[k],errors='coerce')) for k in nums):reason='identical_numeric_column'
            else:nums.append(c)
        if reason:reasons.append({'feature':c,'reason':reason})
    return nums,cats,pd.DataFrame(reasons)
def normalize_inputs(frame,nums,cats):
    d=frame[nums+cats].copy()
    for c in nums:d[c]=pd.to_numeric(d[c],errors='coerce').replace([np.inf,-np.inf],np.nan)
    for c in cats:d[c]=d[c].fillna('__MISSING__').astype(str)
    return d

def fit_cdr(train_raw):
    train,window_report=trend_windows(train_raw)
    if len(train)<3:raise ValueError('Insufficient training windows')
    scaler=StandardScaler().fit(train[TREND_COLUMNS])
    z=scaler.transform(train[TREND_COLUMNS])
    router=MiniBatchKMeans(**MBK_CONFIG).fit(z)
    routes=router.predict(z)
    if len(np.unique(routes))!=3:raise ValueError('Training failed to produce three nonempty regimes')
    nums,cats,reasons=select_columns(train)
    expert_nums=[c for c in nums if c not in TIMING]
    assert not (set(expert_nums+cats)&(set(TIMING)|META|set(TREND_COLUMNS)))
    transformers=[]
    if expert_nums:transformers.append(('numeric',SimpleImputer(strategy='median'),expert_nums))
    if cats:transformers.append(('categorical',OneHotEncoder(handle_unknown='ignore',sparse_output=False),cats))
    pre=ColumnTransformer(transformers,remainder='drop')
    x=pre.fit_transform(normalize_inputs(train,expert_nums,cats))
    experts={}
    for k in range(3):
        mask=routes==k
        experts[k]=RandomForestClassifier(**RF_CONFIG).fit(x[mask],train.loc[mask,'traffic_label'])
    # Cluster numbers remain arbitrary: no mapping to true congestion labels for inference.
    cluster_classes=pd.crosstab(pd.Series(routes,name='cluster'),train.traffic_label).reindex(columns=CLASSES,fill_value=0)
    print('Training cluster/class counts:');display(cluster_classes)
    if (cluster_classes==0).any().any():print('Some experts lack training classes; no oracle fallback is used.')
    print('Expert raw numeric/categorical fields:',len(expert_nums),len(cats),'encoded columns:',x.shape[1])
    sample=min(3000,len(z))
    try: silhouette=silhouette_score(z,routes,sample_size=sample,random_state=SEED)
    except ValueError:silhouette=np.nan
    return dict(router=router,scaler=scaler,pre=pre,experts=experts,nums=expert_nums,cats=cats,
                train=train,routes=routes,window_report=window_report,reasons=reasons,
                cluster_classes=cluster_classes,silhouette=silhouette,encoded_features=pre.get_feature_names_out().tolist())

def predict_cdr(model,test_raw):
    test,report=trend_windows(test_raw)
    routes=model['router'].predict(model['scaler'].transform(test[TREND_COLUMNS]))
    x=model['pre'].transform(normalize_inputs(test,model['nums'],model['cats']))
    pred=np.empty(len(test),dtype=object)
    for k,expert in model['experts'].items():
        mask=routes==k
        if mask.any():pred[mask]=expert.predict(x[mask])
    return test,pred,routes,report

def predict_record(model,stream,sequence_id,record):
    # record contains observed features only; does not read target or congestion labels.
    trend=stream.update(sequence_id,[record[f] for f in TIMING])
    if trend is None:return None
    z=pd.DataFrame([trend],columns=TREND_COLUMNS)
    k=int(model['router'].predict(model['scaler'].transform(z))[0])
    raw=pd.DataFrame([record])
    x=model['pre'].transform(normalize_inputs(raw,model['nums'],model['cats']))
    return str(model['experts'][k].predict(x)[0]),k


## 4. Evaluation helpers
All metrics use the same retained test records. Weighted and macro F1 are both reported. RF references: (a) same expert inputs, 100 trees; (b) those inputs plus the 3 raw timing fields, 100 trees. These are reference RFs, not claimed exact reproductions of all baselines in the paper.

No AF, DFE or CNN result is fabricated. Their faithful replication requires separate method specifications and implementations. No test-set tuning, oracle expert choice, or supervised congestion router is substituted for MBK.


In [ ]:
def evaluate(y,pred,name,folder):
    p,r,f,_=precision_recall_fscore_support(y,pred,average='weighted',zero_division=0)
    macro=precision_recall_fscore_support(y,pred,average='macro',zero_division=0)[2]
    metrics=dict(method=name,accuracy=accuracy_score(y,pred),precision_weighted=p,recall_weighted=r,f1_weighted=f,f1_macro=macro)
    report=classification_report(y,pred,labels=CLASSES,output_dict=True,zero_division=0)
    pd.DataFrame(report).T.to_csv(folder/f'{name}_classification_report.csv')
    cm=confusion_matrix(y,pred,labels=CLASSES)
    pd.DataFrame(cm,index=CLASSES,columns=CLASSES).to_csv(folder/f'{name}_confusion.csv')
    return metrics

def run_scenario(name,train_level,test_level):
    folder=OUT/name;folder.mkdir(parents=True,exist_ok=True)
    tr=data[data.congestion_level.eq(train_level)].copy()
    te=data[data.congestion_level.eq(test_level)].copy()
    assert set(tr.source_file).isdisjoint(te.source_file)
    model=fit_cdr(tr)
    test,pred,routes,test_windows=predict_cdr(model,te)
    # Verify inference does not depend on either target or congestion annotation.
    altered=te.copy();altered['traffic_label']='HIDDEN';altered['congestion_level']='HIDDEN'
    _,pred2,routes2,_=predict_cdr(model,altered)
    assert np.array_equal(pred,pred2) and np.array_equal(routes,routes2)
    # Compare streaming and batch predictions on the first few raw rows of every sequence.
    small=te.groupby('sequence_id',sort=False).head(8)
    bt,bp,br,_=predict_cdr(model,small)
    state=StreamTrend();sp=[];sr=[]
    for _,record in small.iterrows():
        result=predict_record(model,state,record.sequence_id,record.to_dict())
        if result is not None:sp.append(result[0]);sr.append(result[1])
    assert list(bp)==sp and list(br)==sr
    rows=[evaluate(test.traffic_label,pred,'CDR_MLC',folder)]
    if RUN_REFERENCE_RF:
        for include_timing in [False,True]:
            nums=model['nums']+(TIMING if include_timing else [])
            cats=model['cats'];trans=[('numeric',SimpleImputer(strategy='median'),nums)]
            if cats:trans.append(('categorical',OneHotEncoder(handle_unknown='ignore',sparse_output=False),cats))
            baseline=Pipeline([('pre',ColumnTransformer(trans)),('rf',RandomForestClassifier(**{**RF_CONFIG,'n_estimators':100}))])
            baseline.fit(normalize_inputs(model['train'],nums,cats),model['train'].traffic_label)
            bp=baseline.predict(normalize_inputs(test,nums,cats))
            rows.append(evaluate(test.traffic_label,bp,'RF_100_with_timing' if include_timing else 'RF_100_expert_inputs',folder))
    metrics=pd.DataFrame(rows).set_index('method');display(metrics)
    model['window_report'].to_csv(folder/'train_windows.csv',index=False)
    test_windows.to_csv(folder/'test_windows.csv',index=False)
    model['reasons'].to_csv(folder/'excluded_features.csv',index=False)
    model['cluster_classes'].to_csv(folder/'train_cluster_classes.csv')
    metrics.to_csv(folder/'metrics.csv')
    predictions=test[['source_file','source_row','timestamp','traffic_label','congestion_level']].copy()
    predictions['predicted_label']=pred;predictions['cluster']=routes
    predictions.to_csv(folder/'predictions.csv',index=False)
    fig,ax=plt.subplots(figsize=(6,5))
    ConfusionMatrixDisplay.from_predictions(test.traffic_label,pred,labels=CLASSES,normalize='true',values_format='.2f',ax=ax,colorbar=False)
    ax.set_title(f'{name}: {train_level} → {test_level}');fig.tight_layout();fig.savefig(folder/'confusion.png',dpi=150);plt.show()
    manifest=dict(scenario=name,train_level=train_level,test_level=test_level,seed=SEED,window=WINDOW,ddof=0,
       mbk=MBK_CONFIG,rf=RF_CONFIG,expert_numeric=model['nums'],expert_categorical=model['cats'],
       encoded_features=model['encoded_features'],train_records=len(model['train']),test_records=len(test),
       train_silhouette=model['silhouette'],files=audit,sklearn=sklearn.__version__,
       protocol='Paper architecture on regenerated data; documented defaults; no independent server-2 dataset.')
    (folder/'manifest.json').write_text(json.dumps(manifest,indent=2),encoding='utf-8')
    print('Inference label-independence and streaming equivalence passed. Reports:',folder)
    return model,metrics


## Scenario 1 — Low → Medium
Run setup cells above, then run any scenario cell independently. There is no combined scenario runner.


In [ ]:
scenario_1_model, scenario_1_metrics = run_scenario('scenario_1','Low','Medium')


## Scenario 2 — Low → High


In [ ]:
scenario_2_model, scenario_2_metrics = run_scenario('scenario_2','Low','High')


## Scenario 3 — Medium → High


In [ ]:
scenario_3_model, scenario_3_metrics = run_scenario('scenario_3','Medium','High')


## Scenarios 4 and 5 — pending independent server-2 data
Table 4 defines two different collection environments. Splitting or shuffling the present dataset cannot create that second environment. These scenarios are deliberately not replaced with random train/test splits.

## Method fidelity checklist
- Implemented: 15 trailing statistics, MBK with 3 unlabeled clusters, 3 RF multiclass experts with 20 trees, exclusion of timing from experts, frozen train-fitted transforms, inference via predicted cluster.
- Implemented: Table 4 domain directions for scenarios 1–3 on regenerated captures; per-scenario results, manifests and confusion matrices.
- Not identical to published experiments: new collection, unknown original 35-column schema, explicit hyperparameter/window assumptions, ignored invalid IdleTime and unavailable fields, independent-run sequence resets.
- Pending: exact original field list and undocumented parameters, second-server data, separate AF/DFE/CNN reproductions and external dataset experiments.
- Performance is reported as measured. Poor results are not corrected by using test labels or choosing an oracle expert.


## Initial validation run

All 8 code cells executed successfully on 106,512 service records. Batch/streaming inference and independence from test labels passed for all three scenarios. Expert input dimensions were 23 numeric + 3 categorical raw fields (89 encoded columns in scenarios 1/2, 96 in scenario 3); hence the original 35→32 dimension claim is not reproduced with the present schema.

| Scenario | CDR-MLC accuracy | RF same expert inputs | RF plus timing |
|---|---:|---:|---:|
| Low → Medium | 0.901372 | 0.916919 | 0.906715 |
| Low → High | 0.861346 | 0.894298 | 0.872726 |
| Medium → High | 0.889702 | 0.902270 | 0.905771 |

No superiority claim is supported by this initial run. These scores use documented implementation assumptions and the regenerated data, not the original paper experiment. Outputs are cleared in Git; Run All regenerates reports locally.
